# 🫧 25 · Molecules as clouds of Gaussians: single-particle reconstruction and Gaussian mixture models
### From thousands of noisy 2D pictures to a 3D map — and a compact, differentiable way to describe that map

**BiologyToResearch** · Part F — Conformational heterogeneity: Gaussian mixtures and neural fields · Module 25 of 34

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/cryoem/tutorial/BiologyToResearch/notebooks/25_gaussian_mixture_molecules.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 2–3 min to run everything); a GPU is optional |
| **Core frameworks** | Python 3, NumPy, SciPy, Matplotlib, PyTorch |
| **Prerequisites** | [02 · Rotations](02_rotations_geometry.ipynb), [03 · Fourier transform](03_fourier_frequency.ipynb), [04 · Statistics](04_noise_statistics_inference.ipynb) (Gaussian mixtures, K-means), [05 · Deep learning](05_deep_learning_pytorch.ipynb) (gradient descent), [06 · Image formation](06_electron_image_formation.ipynb), [10 · FSC](10_resolution_validation.ipynb) |
| **Estimated time** | 120 min (reading + running + the ✏️ exercises) |
| **Lab papers covered** | Chen & Ludtke 2021 (deep mixed-dimensional GMM, *Nat Methods*) — the representation; Chen, Toader & Lederman 2023 (scalable high-resolution deep GMM, *J Mol Biol*) — separable projection, model-based start; Chen, Schmid & Chiu 2024 (resolution and resolvability with GMMs, *Nat Methods*) — K-means pseudo-atoms, the projection formula — see [`PAPER_ATLAS.md`](../PAPER_ATLAS.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis. All
> experiments are small, controlled *toy versions* of published methods (a 40 × 40-pixel world, one thousand particles,
> a few hundred Gaussians, a minute of training). They illustrate the core ideas; they do **not** reproduce the papers'
> numbers. **All molecules and images are generated by code** (nothing is downloaded): the test protein is built from
> ideal α-helices, and our particle images have no CTF (real images do — see the 🚀 Bonus).

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Describe** a single-particle cryo-EM experiment as data: thousands of noisy 2D projections of one molecule in
   random orientations, and **simulate** such a data set.
2. **Explain** the projection-slice theorem in 3D, **verify** it numerically, and **reconstruct** a 3D map from
   projections with known orientations by inserting Fourier slices; **measure** how the map improves with the number of
   particles.
3. **Represent** a density map as a **Gaussian mixture model (GMM)** — one Gaussian per atom, or a few hundred
   "pseudo-atoms" from weighted K-means — and **measure** how many Gaussians a given resolution needs.
4. **Derive** the projection of a 3D Gaussian, **implement** the fast separable projection of a whole GMM, and check it
   against brute-force rotate-and-sum.
5. **Implement** the Fourier ring correlation (FRC) and **explain** why it makes a good loss for comparing a model with
   noisy, filtered images.
6. **Fit** a GMM directly to noisy particle images by gradient descent (PyTorch autograd), growing the number of
   Gaussians as the resolution improves, and **compare** it with direct Fourier reconstruction.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | tools, the test protein and its true map | 5 min |
| 1 · The single-particle experiment | simulate 1,000 noisy particle images | 10 min |
| 2 · From pictures to a 3D map | projection-slice theorem, Fourier insertion, FSC vs number of particles | 20 min |
| 3 · A molecule as a cloud of Gaussians | one Gaussian per atom; K-means pseudo-atoms; how many Gaussians? | 20 min |
| 4 · Projecting a Gaussian mixture | the 2D Gaussian formula, the separable trick, Fourier space | 15 min |
| 5 · Comparing images: the FRC | why FRC and not squared error | 10 min |
| 6 · Fitting a GMM to particles | autograd, FRC loss, growing the mixture | 25 min |
| 7 · What did the GMM buy us? | GMM fit vs direct reconstruction from the same data | 10 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 10 min |

## 🧫 The biological question

Proteins are machines with moving parts: a kinase closes its two lobes around ATP, a transporter rocks between
inward- and outward-facing states, a ribosome ratchets as it translates. Single-particle cryo-EM freezes millions of
copies of a molecule mid-action and photographs each one. Classical processing averages them all into **one** 3D map —
which blurs exactly the parts that move. The next six modules (Part F) are about recovering the *movie* instead of one
blurred frame. They all need two ingredients that this module builds from scratch: (1) how a 3D map is reconstructed from
2D pictures, and (2) a compact, **differentiable** description of a molecule that a computer can deform — a cloud of
Gaussian blobs. That second idea is the heart of a line of work by Muyuan Chen and colleagues (Chen & Ludtke 2021; Chen,
Toader & Lederman 2023; Chen, Schmid & Chiu 2024; Chen 2025, 2026) that Modules 25–30 follow step by step.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell checks what computer you are on.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, this notebook runs on a CPU.")
try:
    import torch
    print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
except ImportError:
    print("PyTorch not installed yet — the next cell installs it.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("matplotlib", "matplotlib"), ("torch", "torch")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from scipy import ndimage

SEED = 0
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "image.cmap": "gray"})
T_START = time.time()
print("NumPy", np.__version__, "| PyTorch", torch.__version__, "| device:", DEVICE, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py). New today:
`flexible_protein` (a code-built test protein with a movable arm, used in all of Part F), `gmm_to_map` and
`gmm_project` (a Gaussian mixture as a map and as a projection image), `pseudoatoms_kmeans` (Section 3) and
`reconstruct_fourier` (Section 2). Everything else you met earlier: rotations (Module 02), `fsc` (Module 10).

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v3.0; copied so this notebook runs alone) ──
import os
import struct

import numpy as np
from scipy import ndimage

_BOND = dict(N_CA=1.458, CA_C=1.525, C_N=1.329, C_O=1.231)
_ANGLE = dict(N_CA_C=111.2, CA_C_N=116.2, C_N_CA=121.7, CA_C_O=120.5)
_SIDECHAIN = {
    "GLY": [], "ALA": [], "SER": [("OG", "O")], "CYS": [("SG", "S")], "VAL": [("CG1", "C"), ("CG2", "C")],
    "THR": [("OG1", "O"), ("CG2", "C")], "PRO": [("CG", "C"), ("CD", "C")],
    "LEU": [("CG", "C"), ("CD1", "C"), ("CD2", "C")], "ILE": [("CG1", "C"), ("CG2", "C"), ("CD1", "C")],
    "ASP": [("CG", "C"), ("OD1", "O"), ("OD2", "O")], "ASN": [("CG", "C"), ("OD1", "O"), ("ND2", "N")],
    "GLU": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("OE2", "O")], "GLN": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("NE2", "N")],
    "LYS": [("CG", "C"), ("CD", "C"), ("CE", "C"), ("NZ", "N")], "MET": [("CG", "C"), ("SD", "S"), ("CE", "C")],
    "ARG": [("CG", "C"), ("CD", "C"), ("NE", "N"), ("CZ", "C"), ("NH1", "N"), ("NH2", "N")],
    "HIS": [("CG", "C"), ("ND1", "N"), ("CD2", "C"), ("CE1", "C"), ("NE2", "N")],
    "PHE": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C")],
    "TYR": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C"), ("OH", "O")],
    "TRP": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("NE1", "N"), ("CE2", "C"), ("CE3", "C"), ("CZ2", "C"),
            ("CZ3", "C"), ("CH2", "C")],
}
_AA_FREQ = {"LEU": 9.9, "ALA": 8.3, "GLY": 7.1, "VAL": 6.9, "GLU": 6.8, "SER": 6.6, "ILE": 5.9, "LYS": 5.8, "ARG": 5.5,
            "ASP": 5.5, "THR": 5.3, "PRO": 4.7, "ASN": 4.1, "GLN": 3.9, "PHE": 3.9, "TYR": 2.9, "MET": 2.4, "HIS": 2.3,
            "CYS": 1.4, "TRP": 1.1}
def _nerf(a, b, c, bond, angle, torsion):
    """Place atom d so that |cd| = bond, angle(b,c,d) = angle and dihedral(a,b,c,d) = torsion (degrees)."""
    bc = (c - b) / np.linalg.norm(c - b)
    n = np.cross(b - a, bc)
    n /= np.linalg.norm(n)
    m = np.cross(n, bc)
    ang, tor = np.radians(angle), np.radians(torsion)
    d = np.array([-bond * np.cos(ang), bond * np.sin(ang) * np.cos(tor), bond * np.sin(ang) * np.sin(tor)])
    return c + d[0] * bc + d[1] * m + d[2] * n


def build_backbone(phi, psi, omega=None) -> np.ndarray:
    """Backbone atoms N, CA, C, O, CB (array n×5×3, Å) of a chain with the given torsion angles (degrees)."""
    n = len(phi)
    omega = np.full(n, 180.0) if omega is None else np.asarray(omega, float)
    N = [np.zeros(3)]
    CA = [np.array([_BOND["N_CA"], 0.0, 0.0])]
    C = [_nerf(np.array([0.0, 1.0, 0.0]), N[0], CA[0], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[0])]
    for i in range(n - 1):
        N.append(_nerf(N[i], CA[i], C[i], _BOND["C_N"], _ANGLE["CA_C_N"], psi[i]))
        CA.append(_nerf(CA[i], C[i], N[i + 1], _BOND["N_CA"], _ANGLE["C_N_CA"], omega[i]))
        C.append(_nerf(C[i], N[i + 1], CA[i + 1], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[i + 1]))
    N, CA, C = np.array(N), np.array(CA), np.array(C)
    O = np.array([_nerf(N[i], CA[i], C[i], _BOND["C_O"], _ANGLE["CA_C_O"], psi[i] + 180.0) for i in range(n)])
    b, c = CA - N, C - CA
    a = np.cross(b, c)
    CB = -0.58273431 * a + 0.56802827 * b - 0.54067466 * c + CA      # ideal Cβ position
    return np.stack([N, CA, C, O, CB], 1)


def _ideal_helix_template(n=160):
    """An ideal α-helix (φ = −57.8°, ψ = −47.0°) with its axis along +z through x = y = 0."""
    bb = build_backbone(np.full(n, -57.8), np.full(n, -47.0))
    ca = bb[:, 1]
    centre = ca.mean(0)
    u = np.linalg.svd(ca - centre)[2][0]
    if np.dot(ca[-1] - ca[0], u) < 0:
        u = -u
    # rotation taking u to z
    v = np.cross(u, [0, 0, 1.0]); s = np.linalg.norm(v); cth = u[2]
    K = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    Rm = np.eye(3) + K + K @ K * ((1 - cth) / (s ** 2 + 1e-12))
    xyz = (bb - centre) @ Rm.T
    axis_xy = xyz[:, 1, :2].mean(0)
    xyz[..., :2] -= axis_xy
    return xyz


_HELIX = _ideal_helix_template()
def _sidechains(bb, resnames, rng):
    """Approximate side-chain heavy atoms grown outward from Cβ (positions are schematic; counts/elements are right)."""
    names, elems, xyz, resid = [], [], [], []
    ca, cb = bb[:, 1], bb[:, 4]
    out = cb - ca
    out /= np.linalg.norm(out, axis=1, keepdims=True)
    for i, rn in enumerate(resnames):
        sc = _SIDECHAIN[rn]
        if not sc:
            continue
        perp = np.cross(out[i], rng.normal(size=3)); perp /= np.linalg.norm(perp)
        for k, (an, el) in enumerate(sc):
            depth = 1.0 + (k // 2)                               # schematic zig-zag outward
            pos = cb[i] + 1.5 * depth * out[i] + 0.9 * (1 if k % 2 else -1) * perp
            names.append(an); elems.append(el); xyz.append(pos); resid.append(i)
    return names, elems, np.array(xyz).reshape(-1, 3), np.array(resid, int)


def _loop_path(a, b, outward, step=3.8):
    """Cα positions of a loop from Cα `a` to Cα `b`: a circular arc bulging along `outward`, cut into equal chords so
    that every Cα–Cα distance (including the two ends) is exactly `step` Å."""
    c = np.linalg.norm(b - a)
    e = (b - a) / c
    u = outward - (outward @ e) * e                                  # bulge direction ⟂ to the chord
    if np.linalg.norm(u) < 1e-6:
        u = np.cross(e, [0.0, 0.0, 1.0]) if abs(e[2]) < 0.9 else np.cross(e, [1.0, 0.0, 0.0])
    u = u / np.linalg.norm(u)
    m = max(int(np.ceil(c / step)), 1)                               # m new residues → n = m + 1 equal chords
    n = m + 1
    lo, hi = 1e-9, 2 * np.pi - 1e-9                                  # arc angle Θ: sin(Θ/2) / sin(Θ/2n) = c / step
    for _ in range(60):
        th = (lo + hi) / 2
        lo, hi = (th, hi) if np.sin(th / 2) / np.sin(th / (2 * n)) > c / step else (lo, th)
    R = step / (2 * np.sin(th / (2 * n)))
    centre = (a + b) / 2 - R * np.cos(th / 2) * u
    beta = np.pi / 2 + th / 2 - th * np.arange(1, n) / n             # angles of the new points, a → b over the bulge
    return centre + R * (np.outer(np.cos(beta), e) + np.outer(np.sin(beta), u))


def _assemble_chain(ca_path_segments, rng, chain="A", start_resnum=1):
    """Turn helices (n×5×3 arrays) joined by loops into one chain with residue names and side chains."""
    blocks = []
    for k, seg in enumerate(ca_path_segments):
        blocks.append(seg)
        if k + 1 < len(ca_path_segments):
            a, b = seg[-1, 1], ca_path_segments[k + 1][0, 1]
            path = _loop_path(a, b, (a + b) / 2 - seg.reshape(-1, 3).mean(0))
            m = len(path)
            tang = np.gradient(path, axis=0) if m > 1 else np.array([b - a])
            tang /= np.linalg.norm(tang, axis=1, keepdims=True)
            side = np.cross(tang, rng.normal(size=3)); side /= np.linalg.norm(side, axis=1, keepdims=True)
            loop = np.stack([path - 1.2 * tang, path, path + 1.2 * tang, path + 1.2 * tang + 1.0 * side,
                             path - 1.3 * side], 1)
            blocks.append(loop)
    bb = np.concatenate(blocks, 0)
    aa = list(_AA_FREQ)
    p = np.array([_AA_FREQ[a] for a in aa]); p /= p.sum()
    resnames = rng.choice(aa, len(bb), p=p)
    bb_names = ["N", "CA", "C", "O", "CB"]
    names, elems, xyz, resid = [], [], [], []
    for i in range(len(bb)):
        for j, an in enumerate(bb_names):
            if an == "CB" and resnames[i] == "GLY":
                continue
            names.append(an); elems.append(an[0]); xyz.append(bb[i, j]); resid.append(i)
    sn, se, sx, sr = _sidechains(bb, resnames, rng)
    names += sn; elems += se; resid = np.concatenate([np.array(resid, int), sr])
    xyz = np.concatenate([np.array(xyz), sx], 0)
    order = np.argsort(resid, kind="stable")
    return dict(xyz=xyz[order].astype(np.float32), name=np.array(names)[order], element=np.array(elems)[order],
                resname=resnames[resid[order]], resnum=(resid[order] + start_resnum).astype(int),
                chain=np.full(len(order), chain))


def helix_bundle(radii_A, rng, chain="A", spacing=10.0):
    """A globular 'domain' that fills an ellipsoid (semi-axes in Å, long axis = local z) with antiparallel helices."""
    a, b, c = radii_A[0] + 2.5, radii_A[1] + 2.5, radii_A[2] + 1.5   # helices stop short of the surface: pad
    rows = []
    j = 0
    while (j * spacing * np.sqrt(3) / 2) <= b:
        rows.append(j); rows.append(-j) if j else None; j += 1
    pts = []
    for jj in sorted(set(rows)):
        y = jj * spacing * np.sqrt(3) / 2
        xs = np.arange(-a, a + 1e-6, spacing) + (spacing / 2 if jj % 2 else 0)
        row = [(x, y) for x in xs if (x / a) ** 2 + (y / b) ** 2 < 1]
        pts.append(row if (jj % 2 == 0) else row[::-1])
    pts = [p for row in pts for p in row]
    segs = []
    for k, (x, y) in enumerate(pts):
        h = c * np.sqrt(max(0.0, 1 - (x / a) ** 2 - (y / b) ** 2)) - 2.0
        n = int(2 * h / 1.5)
        if n < 6:
            continue
        seg = _HELIX[:n].copy()
        seg[..., 2] -= seg[:, 1, 2].mean()
        phase = rng.uniform(0, 2 * np.pi)
        Rz = np.array([[np.cos(phase), -np.sin(phase), 0], [np.sin(phase), np.cos(phase), 0], [0, 0, 1]])
        seg = seg @ Rz.T
        if len(segs) % 2 == 1:                                   # antiparallel neighbours: 180° turn about x
            seg = seg * np.array([1.0, -1.0, -1.0])
        seg = seg + np.array([x, y, rng.uniform(-1.0, 1.0)])
        segs.append(seg)
    return _assemble_chain(segs, rng, chain)


def _transform(atoms, R=np.eye(3), t=(0, 0, 0)):
    out = dict(atoms)
    out["xyz"] = (atoms["xyz"] @ np.asarray(R, float).T + np.asarray(t, float)).astype(np.float32)
    return out


def merge_atoms(parts):
    keys = parts[0].keys()
    return {k: np.concatenate([p[k] for p in parts]) for k in keys}


def _rz(a):
    return np.array([[np.cos(a), -np.sin(a), 0], [np.sin(a), np.cos(a), 0], [0, 0, 1.0]])


def random_rotation(rng: np.random.Generator) -> np.ndarray:
    """Uniformly random proper rotation (det = +1) from a random unit quaternion."""
    q = rng.normal(size=4)
    q /= np.linalg.norm(q)
    w, x, y, z = q
    return np.array([
        [1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
        [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
        [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])


_P = np.array([[0, 0, 1], [0, 1, 0], [1, 0, 0]], float)  # (x,y,z) <-> (z,y,x)
def rotate_volume(vol: np.ndarray, R: np.ndarray, shift_xyz=(0, 0, 0), order: int = 1,
                  cval: float = 0.0) -> np.ndarray:
    """Rotate a volume about its centre by R (then translate by shift, in voxels, (x,y,z)).

    out(p) = vol(Rᵀ (p − c − t) + c): the *object* is rotated by R and moved by t.
    """
    R_idx = _P @ R @ _P                       # same rotation written in (z, y, x) index order
    c = (np.array(vol.shape) - 1) / 2.0
    t_idx = _P @ np.asarray(shift_xyz, float)
    M = R_idx.T                               # inverse rotation maps output -> input
    offset = c - M @ (c + t_idx)
    return ndimage.affine_transform(vol, M, offset=offset, order=order, mode="constant", cval=cval)


def freq_grid(shape):
    """Frequency coordinates (cycles/voxel) for an unshifted FFT; returns list per axis."""
    return np.meshgrid(*[np.fft.fftfreq(n) for n in shape], indexing="ij")


def radial_frequency(shape) -> np.ndarray:
    """|k| in cycles/voxel for every coefficient of an unshifted FFT."""
    return np.sqrt(sum(k**2 for k in freq_grid(shape)))


def apply_fourier_mask(vol: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Multiply the FFT of ``vol`` by ``mask`` (unshifted layout) and return the real part."""
    return np.real(np.fft.ifftn(np.fft.fftn(vol) * mask)).astype(np.float32)


def lowpass(vol: np.ndarray, cutoff: float, soft: float = 0.02) -> np.ndarray:
    """Soft low-pass filter; ``cutoff`` in cycles/voxel (0.5 = Nyquist)."""
    k = radial_frequency(vol.shape)
    filt = 1 / (1 + np.exp((k - cutoff) / soft))
    return apply_fourier_mask(vol, filt)


def pearson(a: np.ndarray, b: np.ndarray) -> float:
    """Pearson correlation of two arrays (flattened)."""
    a = a.ravel() - a.mean()
    b = b.ravel() - b.mean()
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))


def fsc(v1: np.ndarray, v2: np.ndarray, pixel_A: float = 10.0, n_shells: int | None = None):
    """Fourier shell (ring, in 2D) correlation. Returns (frequency_1/Å, fsc, n_coeffs)."""
    F1, F2 = np.fft.fftn(v1), np.fft.fftn(v2)
    k = radial_frequency(v1.shape)
    n_shells = n_shells or min(v1.shape) // 2
    idx = np.minimum((k * 2 * n_shells).astype(int), n_shells)
    num = np.bincount(idx.ravel(), (F1 * np.conj(F2)).real.ravel(), n_shells + 1)
    d1 = np.bincount(idx.ravel(), (np.abs(F1)**2).ravel(), n_shells + 1)
    d2 = np.bincount(idx.ravel(), (np.abs(F2)**2).ravel(), n_shells + 1)
    cnt = np.bincount(idx.ravel(), minlength=n_shells + 1)
    curve = num / np.sqrt(d1 * d2 + 1e-20)
    freq = (np.arange(n_shells + 1) + 0.5) / (2 * n_shells) / pixel_A
    return freq[:n_shells], curve[:n_shells], cnt[:n_shells]


def resolution_at(freq, curve, threshold: float = 0.143) -> float:
    """First crossing of the FSC curve below ``threshold`` → resolution in Å."""
    below = np.where(curve[1:] < threshold)[0]
    if len(below) == 0:
        return float(1 / freq[-1])
    i = below[0] + 1
    f0, f1, c0, c1 = freq[i - 1], freq[i], curve[i - 1], curve[i]
    f = f0 + (threshold - c0) * (f1 - f0) / (c1 - c0 + 1e-12)
    return float(1 / f)


def flexible_protein(seed: int = 7):
    """A code-generated ~80 kDa protein for heterogeneity experiments (Modules 25–30).

    Domains 0 and 1 form a rigid body; domain 2 (the 'arm') sits on domain 1 and can swing about a hinge; domain 3 is
    a small partner subunit that may be bound or not. Returns (atoms, hinge (x, y, z) in Å); atoms['domain'] labels
    every atom. Coordinates are in Å, centred near the origin.
    """
    rng = np.random.default_rng(seed)
    specs = [((21.0, 17.5, 16.0), (-19.0, -7.5, 0.0)), ((19.0, 16.0, 17.5), (19.0, -7.5, 0.0)),
             ((17.5, 12.5, 14.0), (21.0, 33.0, 0.0)), ((12.5, 11.0, 11.0), (-21.0, 26.0, 2.5))]
    parts = []
    for d, (radii, centre) in enumerate(specs):
        part = _transform(helix_bundle(radii, rng, chain="ABCD"[d]), _rz(rng.uniform(0, 2 * np.pi)), centre)
        part["domain"] = np.full(len(part["xyz"]), d)
        parts.append(part)
    atoms = merge_atoms(parts)
    shift = atoms["xyz"][atoms["domain"] < 3].mean(0)
    atoms["xyz"] = (atoms["xyz"] - shift).astype(np.float32)
    return atoms, np.array([19.0, 14.0, 0.0]) - shift


def pseudoatoms_kmeans(vol: np.ndarray, n: int, voxel: float = 1.0, rng=None, iters: int = 30,
                       level: float = 0.05):
    """Summarise a density map by n Gaussian 'pseudo-atoms': weighted K-means on the voxels above level × max.

    Returns centres (n, 3) as (x, y, z) in Å relative to the box centre, and amplitudes (density in each cluster).
    """
    rng = np.random.default_rng(0) if rng is None else rng
    idx = np.argwhere(vol > level * vol.max())                                 # (z, y, x) of dense voxels
    w = vol[tuple(idx.T)].astype(np.float64)
    pts = ((idx[:, ::-1] - (np.array(vol.shape[::-1]) - 1) / 2.0) * voxel).astype(np.float32)
    c = pts[rng.choice(len(pts), n, replace=False, p=w / w.sum())].copy()
    for _ in range(iters):
        d2 = (pts ** 2).sum(1)[:, None] - 2 * pts @ c.T + (c ** 2).sum(1)[None]
        lab = d2.argmin(1)
        ws = np.bincount(lab, w, n)
        for a in range(3):
            upd = np.bincount(lab, w * pts[:, a], n)
            c[ws > 0, a] = upd[ws > 0] / ws[ws > 0]
    return c.astype(np.float32), np.bincount(lab, w, n).astype(np.float32)


def gmm_to_map(centres, amps, sigmas, box: int, voxel: float = 1.0) -> np.ndarray:
    """Density [z, y, x] (box³, origin at the box centre) of Σ_j A_j exp(−|x − c_j|² / 2σ_j²); c and σ in Å."""
    centres = np.asarray(centres, float)
    s = np.broadcast_to(np.asarray(sigmas, float), (len(centres),))
    g = (np.arange(box) - (box - 1) / 2.0) * voxel
    ex = [np.exp(-(g[None, :] - centres[:, a, None]) ** 2 / (2 * s[:, None] ** 2)) for a in range(3)]
    return np.einsum("n,nz,ny,nx->zyx", np.asarray(amps, float), ex[2], ex[1], ex[0], optimize=True).astype(np.float32)


def gmm_project(centres, amps, sigmas, R, box: int, pixel: float = 1.0) -> np.ndarray:
    """Projection [y, x] along z of a Gaussian mixture rotated by R (acting on (x, y, z)); same units as `gmm_to_map`.

    A 3D Gaussian projects to a 2D Gaussian of the same width, so the image is Σ_j w_j · g_y,j ⊗ g_x,j — two small
    matrices instead of a 3D grid (the 'project x and y separately' trick).
    """
    p = (np.asarray(centres, np.float32) @ np.asarray(R, np.float32).T)
    s = np.broadcast_to(np.asarray(sigmas, np.float32), (len(p),))[:, None]
    g = ((np.arange(box) - (box - 1) / 2.0) * pixel).astype(np.float32)[None, :]
    gx = np.exp(-(g - p[:, 0, None]) ** 2 / (2 * s ** 2))
    gy = np.exp(-(g - p[:, 1, None]) ** 2 / (2 * s ** 2))
    w = np.asarray(amps, np.float32)[:, None] * np.float32(np.sqrt(2 * np.pi)) * s / np.float32(pixel)   # ∫ dz
    return ((gy * w).T @ gx).astype(np.float32)


def reconstruct_fourier(images: np.ndarray, rotations, wiener: float = 1.0, chunk: int = 64) -> np.ndarray:
    """3D map [z, y, x] from projection images [n, y, x] with known rotations (R acts on (x, y, z), as in
    `rotate_volume`; image = rotate_volume(vol, R).sum(axis=0)).

    Projection-slice theorem: each image's 2D Fourier transform is the central slice of the 3D transform at
    k₃ = Rᵀ(kx, ky, 0). Slices are inserted with trilinear weights into a 2× padded grid, divided by the summed
    weights (+ `wiener`, in units of one image), transformed back and corrected for the interpolation kernel.
    """
    images = np.asarray(images, np.float64)
    n, N, _ = images.shape
    M, pad = 2 * N, N // 2
    k = np.arange(M) - M // 2
    ky, kx = np.meshgrid(k, k, indexing="ij")
    keep = (kx ** 2 + ky ** 2).ravel() <= (M // 2 - 1) ** 2
    plane = np.stack([kx.ravel(), ky.ravel(), np.zeros(M * M)], 1)[keep]
    shift_phase = np.exp(-1j * np.pi * (kx + ky) / M).ravel()[keep]          # half-pixel: image centre → index M/2
    Fr, Fi, W = np.zeros(M ** 3), np.zeros(M ** 3), np.zeros(M ** 3)
    corners = [(dz, dy, dx) for dz in (0, 1) for dy in (0, 1) for dx in (0, 1)]
    for start in range(0, n, chunk):
        idx_all, val_all, w_all = [], [], []
        for img, R in zip(images[start:start + chunk], list(rotations)[start:start + chunk]):
            big = np.zeros((M, M)); big[pad:pad + N, pad:pad + N] = img
            Fimg = np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(big))).ravel()[keep] * shift_phase
            q = plane @ np.asarray(R, float)                                    # rows: (Rᵀ k)ᵀ in (x, y, z)
            pz = q[:, ::-1] + M // 2                                            # (z, y, x) grid coordinates
            i0 = np.floor(pz).astype(int); f = pz - i0
            for dz, dy, dx in corners:
                ii = i0 + (dz, dy, dx)
                wt = (np.where(dz, f[:, 0], 1 - f[:, 0]) * np.where(dy, f[:, 1], 1 - f[:, 1])
                      * np.where(dx, f[:, 2], 1 - f[:, 2]))
                ok = np.all((ii >= 0) & (ii < M), 1)
                idx_all.append((ii[ok, 0] * M + ii[ok, 1]) * M + ii[ok, 2]); val_all.append(wt[ok] * Fimg[ok])
                w_all.append(wt[ok])
        idx, val, wts = np.concatenate(idx_all), np.concatenate(val_all), np.concatenate(w_all)
        Fr += np.bincount(idx, val.real, M ** 3); Fi += np.bincount(idx, val.imag, M ** 3); W += np.bincount(idx, wts, M ** 3)
    F3 = ((Fr + 1j * Fi) / (W + wiener)).reshape(M, M, M)
    kz3, ky3, kx3 = np.meshgrid(k, k, k, indexing="ij")
    F3 *= np.exp(1j * np.pi * (kx3 + ky3 + kz3) / M)                            # centre back to (N − 1)/2
    vol = np.real(np.fft.fftshift(np.fft.ifftn(np.fft.ifftshift(F3))))[pad:pad + N, pad:pad + N, pad:pad + N]
    x = (np.arange(N) - (N - 1) / 2.0) / M
    sinc2 = np.sinc(x) ** 2                                                     # trilinear-kernel correction
    return (vol / (sinc2[:, None, None] * sinc2[None, :, None] * sinc2[None, None, :])).astype(np.float32)


def show_orthoslices(vol, title: str = "", axes=None, cmap: str = "gray"):
    """Show the three central orthogonal slices (xy, xz, yz) of a volume."""
    import matplotlib.pyplot as plt
    if axes is None:
        _, axes = plt.subplots(1, 3, figsize=(7.5, 2.6))
    z, y, x = (np.array(vol.shape) // 2)
    for ax, im, lab in zip(axes, [vol[z], vol[:, y, :], vol[:, :, x]], ["xy (z mid)", "xz (y mid)", "yz (x mid)"]):
        ax.imshow(im, cmap=cmap)
        ax.set_title(f"{title} {lab}".strip(), fontsize=8)
        ax.axis("off")
    return axes
print("🧰 toolkit loaded")

### Data: a code-built protein and its true density map

Our test molecule (`flexible_protein`) is a ~80 kDa protein built atom by atom from ideal α-helices: two globular
domains form a rigid **body**, a third domain (the **arm**) sits on top of one of them and will swing about a
**hinge** in Module 26, and a small **partner** subunit binds the other. Today it holds still. We describe every atom
by a small Gaussian blob (width σ = 2.5 Å) weighted by its number of electrons, and sample the result on a 40³ grid of
4 Å voxels (a 160 Å box) — the **true map** that our reconstructions will be judged against.

In [ ]:
# ── Data: the test protein and its true map ──
N, PX = 40, 4.0                                   # box size (pixels) and pixel size (Å): a 160 Å field of view
SIG_ATOM = 2.5                                    # Å: blur of one atom (thermal motion + a little resolution loss)
atoms, hinge = flexible_protein()
Z_ATOM = np.array([{"C": 6.0, "N": 7.0, "O": 8.0, "S": 16.0}.get(e, 6.0) for e in atoms["element"]], np.float32)
truth = gmm_to_map(atoms["xyz"], Z_ATOM, SIG_ATOM, N, PX)
DOMAIN_NAMES = ["body 1", "body 2", "arm", "partner"]
for d, name in enumerate(DOMAIN_NAMES):
    sel = atoms["domain"] == d
    print(f"domain {d} ({name:7s}): {sel.sum():5d} atoms, centre at {np.round(atoms['xyz'][sel].mean(0), 1)} Å")
print(f"{len(Z_ATOM):,} atoms in total; molecule extent {np.round(np.ptp(atoms['xyz'], 0), 0)} Å; "
      f"true map {truth.shape} at {PX} Å/voxel; Nyquist {2 * PX:.0f} Å")

fig, axes = plt.subplots(1, 4, figsize=(11, 2.8))
for d, name in enumerate(DOMAIN_NAMES):
    sel = atoms["domain"] == d
    axes[0].scatter(atoms["xyz"][sel, 0], atoms["xyz"][sel, 1], s=0.3, label=name)
axes[0].set_aspect("equal"); axes[0].legend(fontsize=6, markerscale=8, loc="lower left")
axes[0].scatter(*hinge[:2], marker="x", color="k", s=60)
axes[0].invert_yaxis()                                   # y pointing down, like the image panels
axes[0].set_title("atoms (× = hinge)"); axes[0].set_xlabel("x (Å)"); axes[0].set_ylabel("y (Å)")
show_orthoslices(truth, "true map", axes=axes[1:])
plt.tight_layout(); plt.show()

---
## 1 · The single-particle experiment

> 📍 **Where we are.** Module 06 showed how one electron image of a particle forms (a projection, blurred by the CTF,
> drowned in shot noise) and Module 10 how to measure the quality of a 3D map. This module connects them: many noisy
> 2D projections → one 3D map. Modules 26–30 then let the map *move*.

🧫 In a single-particle experiment, a purified protein solution is frozen in a thin film of ice. The copies lie in
**random orientations**, each one is a **projection** (the density summed along the beam, Module 06), and each image is
extremely **noisy** because the electron dose must stay low. Software first finds the particles, then estimates each
particle's orientation — its **pose**, a rotation $R_i$ (Module 02) — by comparing it with a reference ("consensus
refinement"). Pose estimation is a big topic of its own; throughout Part F we assume the poses are known, exactly as the
papers we follow do (they take poses from a standard refinement program).

🧮 **The image model.** Particle $i$ is

$$X_i(x, y) = \int V\big(R_i^{\top}\,(x, y, z)\big)\,dz \;+\; \text{noise}_i(x, y),$$

*in words:* turn the molecule by $R_i$, add up the density along the beam direction $z$, and add noise. (We leave out
the CTF of Module 06 — the papers correct its phases first — and the small in-plane shifts.)

In [ ]:
# ── 1.1 Simulate 1,000 particle images in random orientations ──
N_PART, SNR = 1000, 0.1
rot_true = np.array([random_rotation(rng) for _ in range(N_PART)]).astype(np.float32)
t0 = time.time()
clean = np.array([gmm_project(atoms["xyz"], Z_ATOM, SIG_ATOM, R, N, PX) for R in rot_true])
noise_sd = float(np.sqrt(clean.var() / SNR))                # SNR = variance of signal / variance of noise
particles = (clean + rng.normal(0, noise_sd, clean.shape)).astype(np.float32)
print(f"{N_PART} particles of {N}×{N} pixels in {time.time() - t0:.1f} s; per-image SNR {SNR}")

fig, axes = plt.subplots(2, 6, figsize=(10, 3.6))
for j in range(6):
    axes[0, j].imshow(clean[j]); axes[1, j].imshow(particles[j])
    axes[0, j].set_title(f"particle {j}", fontsize=8)
for ax in axes.ravel():
    ax.axis("off")
axes[0, 0].text(-6, 20, "clean", rotation=90, va="center", fontsize=8)
axes[1, 0].text(-6, 20, "recorded", rotation=90, va="center", fontsize=8)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Each clean projection shows the molecule from a different direction; after adding noise at
SNR 0.1 you can still guess where the particle is, but not its details. (Real cryo-EM particle images are noisier still,
SNR ≈ 0.01–0.1, which is why real data sets have 10⁵–10⁶ particles.)

✏️ **Your turn.** Change `SNR = 0.1` to `0.02` and re-run 1.1 (and later sections). How many more particles would you
need for the same result? *(Hint: Module 04 — noise variance falls as 1/N when you average.)*

---
## 2 · From 2D pictures to a 3D map

### 2.1 The projection-slice theorem in 3D

Module 07 proved it in 2D for tomography: the Fourier transform of a projection is a **central slice** through the
Fourier transform of the object. In 3D:

$$\mathcal F_{2}[X_i](k_x, k_y) \;=\; \mathcal F_{3}[V]\big(R_i^{\top}\,(k_x, k_y, 0)\big),$$

*in words:* every particle image fills one plane through the centre of the molecule's 3D Fourier space, and the plane is
tilted by the particle's pose. Many particles in many orientations fill the whole space; an inverse 3D Fourier
transform then gives the map. Let us check the simplest case numerically: for $R = I$ (no rotation) the theorem says
the 2D transform of the projection along z equals the $k_z = 0$ plane of the 3D transform — exactly, even on a grid.

In [ ]:
# ── 2.1 Check the projection-slice theorem for R = identity ──
proj_z = truth.sum(axis=0)                                    # projection along z (axis 0 of [z, y, x])
F2 = np.fft.fft2(proj_z)
F3 = np.fft.fftn(truth)
print(f"max |FFT2(projection) − (k_z = 0 plane of FFT3)| = {np.abs(F2 - F3[0]).max():.2e}  "
      f"(relative to max |F| = {np.abs(F2).max():.1f})")
assert np.allclose(F2, F3[0], rtol=1e-4, atol=1e-3 * np.abs(F2).max())

### 2.2 Reconstruction by Fourier insertion

For tilted planes, the plane's points fall *between* the grid points of 3D Fourier space. The toolkit's
`reconstruct_fourier` does what classic programs do: for each image it computes the 3D frequency coordinates of its
plane, spreads each Fourier value onto the 8 nearest grid points (**trilinear insertion**, on a grid twice as fine),
keeps a running sum of the weights, divides, and transforms back. Let us reconstruct from the first 20, 100 and 1,000
particles and compare each map with the truth by FSC (Module 10; we use the 0.5 threshold because we compare with a
known truth).

In [ ]:
# ── 2.2 Reconstruct from 20, 100 and 1,000 particles ──
recs, res_direct = {}, {}
for n in [20, 100, 1000]:
    t0 = time.time()
    recs[n] = reconstruct_fourier(particles[:n], rot_true[:n])
    f, c, _ = fsc(recs[n], truth, PX)
    res_direct[n] = resolution_at(f, c, 0.5)
    print(f"{n:5d} particles: reconstructed in {time.time() - t0:4.1f} s; FSC(0.5) with the truth → {res_direct[n]:5.1f} Å")

fig, axes = plt.subplots(1, 4, figsize=(10, 2.8))
axes[0].imshow(truth[N // 2]); axes[0].set_title("truth (central slice)", fontsize=9)
for ax, n in zip(axes[1:], recs):
    ax.imshow(recs[n][N // 2]); ax.set_title(f"{n} particles: {res_direct[n]:.0f} Å", fontsize=9)
for ax in axes:
    ax.axis("off")
plt.tight_layout(); plt.show()
assert res_direct[1000] < res_direct[20], "more particles should give a better map"

🔍 **What just happened?** With 20 particles Fourier space is filled by only 20 planes — the map is mostly noise and
streaks. With 1,000 particles every region of Fourier space has been sampled many times, the noise averages down
(Module 04) and the molecule appears, at a resolution of roughly 15 Å here. The map is still noisy: we treated every
voxel as an independent unknown — 64,000 numbers estimated from noisy data. **Can a smarter description of the molecule
do better with the same particles?** That is where Gaussians come in.

---
## 3 · A molecule as a cloud of Gaussians

A **Gaussian mixture model** (Module 04) describes a density as a sum of $n$ blobs:

$$M(\mathbf x) \;=\; \sum_{j=1}^{n} A_j\,\exp\!\Big(-\frac{\lVert \mathbf x-\mathbf c_j\rVert^2}{2\sigma_j^2}\Big),$$

*in words:* Gaussian $j$ has a centre $\mathbf c_j$ (3 numbers), an amplitude $A_j$ (how much density) and a width
$\sigma_j$ — five numbers per Gaussian. Two ways to choose the Gaussians:

- **One Gaussian per atom.** If an atomic model exists, put a Gaussian on every (non-hydrogen) atom. That is literally how
  we made our true map, and it is how Chen (2025) represents a model (Module 30).
- **Pseudo-atoms from the map.** Without a model, run a **weighted K-means** (Module 04) on the voxels of a map: each
  cluster centre becomes a Gaussian, its amplitude the density in the cluster. Chen, Schmid & Chiu (2024) start this way,
  with about as many Gaussians as the protein has atoms for near-atomic maps.

**How many Gaussians are needed?** Fewer Gaussians mean fewer parameters (easier to estimate from noisy data, easier to
animate) but less detail. Let us measure it.

In [ ]:
# ── 3.1 Pseudo-atoms by K-means: how many Gaussians for which resolution? ──
pseudo = {}
for n_g in [16, 64, 256, 1024]:
    t0 = time.time()
    cen, amp = pseudoatoms_kmeans(truth, n_g, voxel=PX, rng=np.random.default_rng(1))
    best = None
    for s in [2.0, 3.0, 4.0, 6.0, 8.0, 10.0, 12.0]:            # choose the width that best matches the map
        gm = gmm_to_map(cen, amp, s, N, PX)
        f, c, _ = fsc(gm, truth, PX)
        r = resolution_at(f, c, 0.5)
        if best is None or r < best[1]:
            best = (s, r, gm)
    pseudo[n_g] = (cen, amp) + best
    print(f"{n_g:5d} Gaussians (σ = {best[0]:4.1f} Å): FSC(0.5) with the truth → {best[1]:5.1f} Å   "
          f"[{time.time() - t0:.1f} s]")

fig, axes = plt.subplots(1, 4, figsize=(10.5, 2.9))
for ax, (n_g, (cen, amp, s, r, gm)) in zip(axes, pseudo.items()):
    ax.imshow(gm.sum(axis=0), extent=[-N * PX / 2, N * PX / 2, N * PX / 2, -N * PX / 2])
    ax.scatter(cen[:, 0], cen[:, 1], s=4 if n_g < 1000 else 1, color="tab:orange")
    ax.set_title(f"{n_g} Gaussians → {r:.0f} Å", fontsize=9); ax.axis("off")
plt.suptitle("K-means pseudo-atoms (orange) and the GMM map they define (projection along z)", fontsize=10)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Sixteen Gaussians capture the overall shape (≈ 20 Å); every fourfold increase buys a few
ångströms; ~1,000 Gaussians reach the 8 Å Nyquist limit of our grid. Our protein has about 5,400 atoms, so near-atomic
detail would need thousands of Gaussians — the regime of Chen, Schmid & Chiu (2024). Chen & Ludtke (2021) work in the
middle ground: they start with a few hundred Gaussians at low resolution and **double** the number while the fit keeps
improving, ending with a few thousand. A rule of thumb you can now justify: *the number of Gaussians needed grows like
(size / resolution)³*, the number of resolution-sized cells that fit in the molecule.

✏️ **Your turn.** Add `4096` to the list. Does the resolution still improve, or have we hit the limit of the 4 Å grid?

---
## 4 · Projecting a Gaussian mixture

To compare a GMM with particle images we must **project** it at each particle's pose. With a voxel map we would rotate
64,000 voxels (with interpolation) and sum them. A Gaussian is much kinder.

🧮 **Projecting one Gaussian.** Rotate the centre: $\mathbf p = R\,\mathbf c$ (a Gaussian is round, so only its centre
moves). Then integrate along z:

$$\int A\, e^{-\frac{(x-p_x)^2+(y-p_y)^2+(z-p_z)^2}{2\sigma^2}}\,dz \;=\; A\sqrt{2\pi}\,\sigma\;
  e^{-\frac{(x-p_x)^2+(y-p_y)^2}{2\sigma^2}},$$

*in words:* the shadow of a 3D Gaussian is a 2D Gaussian of the same width, centred under it, with weight
$A\sqrt{2\pi}\sigma$ (the integral $\int e^{-z^2/2\sigma^2}dz = \sqrt{2\pi}\sigma$). The projection of the mixture is
$P(x, y)=\sum_j A_j\sqrt{2\pi}\sigma_j\,g_j(x)\,g_j(y)$.

💡 **The separable trick.** Each 2D Gaussian is a product of a function of $x$ and a function of $y$. So we compute two
small tables — $g_j(x)$ for every Gaussian and pixel column ($n \times 40$) and $g_j(y)$ ($n \times 40$) — and one
matrix product gives the whole image. Chen, Toader & Lederman (2023) use exactly this to fit tens of thousands of
Gaussians in GPU memory. The toolkit's `gmm_project` does it.

In [ ]:
# ── 4.1 Analytic GMM projection vs rotate-and-sum of the GMM's voxel map ──
cen256, amp256, s256 = pseudo[256][0], pseudo[256][1], pseudo[256][2]
gmm_map256 = gmm_to_map(cen256, amp256, s256, N, PX)
R_test = random_rotation(np.random.default_rng(7))
t0 = time.time()
for _ in range(20):
    p_analytic = gmm_project(cen256, amp256, s256, R_test, N, PX)
t_analytic = (time.time() - t0) / 20
t0 = time.time()
for _ in range(20):
    p_voxel = rotate_volume(gmm_map256, R_test, order=1).sum(axis=0)
t_voxel = (time.time() - t0) / 20
print(f"correlation analytic vs voxel projection: {pearson(p_analytic, p_voxel):.4f}; "
      f"total intensity ratio {p_analytic.sum() / p_voxel.sum():.4f}")
print(f"time per projection: analytic {1e3 * t_analytic:.2f} ms, rotate + sum {1e3 * t_voxel:.2f} ms")
assert pearson(p_analytic, p_voxel) > 0.99

fig, axes = plt.subplots(1, 3, figsize=(8, 2.7))
for ax, im, t in zip(axes, [p_analytic, p_voxel, p_analytic - p_voxel],
                     ["analytic (separable)", "rotate voxels + sum", "difference"]):
    ax.imshow(im); ax.set_title(t, fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The formula and brute force agree (the small difference is the interpolation blur of
rotating voxels, Module 02). The analytic projection needs no interpolation and no 3D grid at all — and, crucially for
Section 6, it is a smooth function of every centre, amplitude and width, so a computer can take its **derivative**.

### 4.2 The same projection in Fourier space

EMAN2's original `e2gmm_refine.py` computes the projection directly in Fourier space. The 2D Fourier transform of a
Gaussian blob is again a Gaussian, times a phase ramp that encodes its position (the shift theorem of Module 03):

$$\hat P(\mathbf k) \;=\; \sum_j A_j\,\sqrt{2\pi}\sigma_j\;2\pi\sigma_j^2\,e^{-2\pi^2\sigma_j^2\lVert\mathbf k\rVert^2}\,
  e^{-2\pi i\,\mathbf k\cdot\mathbf p_j},$$

*in words:* wider blobs lose their high frequencies faster, and each blob's position is a twist of the phase. Let us
check it against the FFT of our image (up to the pixel-size factor of the discrete transform).

In [ ]:
# ── 4.2 Fourier-space projection vs the FFT of the real-space projection ──
p_rot = cen256 @ R_test.T                                     # rotated centres (x, y, z), Å
kx = np.fft.fftfreq(N, d=PX)                                  # cycles per Å
KY, KX = np.meshgrid(kx, kx, indexing="ij")
w_proj = amp256 * np.sqrt(2 * np.pi) * s256 / PX              # weight of each 2D Gaussian (as in gmm_project)
F_analytic = sum(w * 2 * np.pi * s256 ** 2 / PX ** 2 * np.exp(-2 * np.pi ** 2 * s256 ** 2 * (KX ** 2 + KY ** 2))
                 * np.exp(-2j * np.pi * (KX * (p[0] + (N - 1) / 2 * PX) + KY * (p[1] + (N - 1) / 2 * PX)))
                 for w, p in zip(w_proj, p_rot))
F_fft = np.fft.fft2(p_analytic)
low = np.sqrt(KX ** 2 + KY ** 2) < 1 / (3 * PX)               # compare below 12 Å (the FFT of a sampled image aliases above)
print(f"relative difference (frequencies below 1/12 Å⁻¹): "
      f"{np.abs(F_analytic - F_fft)[low].max() / np.abs(F_fft).max():.2e}")

🔍 The two agree to a small fraction of a per cent at low and medium frequencies (near Nyquist the sampled image picks
up a little aliasing that the continuous formula does not have). Real-space and Fourier-space projections are two views
of one calculation; programs choose whichever is faster on their hardware.

---
## 5 · Comparing images: the Fourier ring correlation

To fit a model we need a **loss**: a number that says how badly a model projection matches a particle. The obvious choice,
the squared difference $\sum (P - X)^2$, has two problems with real data: it depends on the overall brightness (the
microscope's gain, ice thickness) and on how each frequency was filtered (the CTF envelope, Module 06). Chen & Ludtke
(2021) use instead the **Fourier ring correlation** — the 2D version of the FSC of Module 10, averaged over rings:

$$\mathrm{FRC}(P, X) \;=\; \frac{1}{K}\sum_{r=r_{\min}}^{r_{\max}}
  \frac{\sum_{\mathbf k\in r}\operatorname{Re}\,\hat P(\mathbf k)\hat X^*(\mathbf k)}
       {\sqrt{\sum_{\mathbf k\in r}|\hat P(\mathbf k)|^2\;\sum_{\mathbf k\in r}|\hat X(\mathbf k)|^2}},$$

*in words:* within each ring of frequencies, compute the correlation of the two images (a number between −1 and 1),
then average the rings from $r_{\min}$ to a chosen resolution limit $r_{\max}$. Each ring is normalised separately, so
multiplying a whole ring by a constant — brightness, filtering, an envelope — does not change it. (The EMAN2 code skips
the lowest rings, which carry little information about shape.)

In [ ]:
# ── 5.1 FRC vs squared error under brightness changes, filtering and noise ──
def frc_score(a: np.ndarray, b: np.ndarray, rmin: int = 1, rmax: int = N // 2) -> float:
    """Mean Fourier ring correlation over rings rmin..rmax (each ring normalised separately)."""
    _, curve, _ = fsc(a, b, PX)
    return float(curve[rmin:rmax + 1].mean())


ref = clean[0] / clean[0].std()
variants = {"same image × 3": 3 * ref,
            "low-pass filtered (20 Å)": lowpass(ref, PX / 20),
            "noisy copy (SNR 1)": ref + np.random.default_rng(3).normal(0, 1.0, ref.shape),
            "a different view": clean[1] / clean[1].std()}
print(f"{'compared with the reference':28s}  {'squared error':>13s}  {'FRC (rings 1–15)':>16s}")
for name, v in variants.items():
    print(f"{name:28s}  {np.mean((v - ref) ** 2):13.3f}  {frc_score(ref, v, 1, 15):16.3f}")

🔍 **What just happened?** Squared error punishes a merely *brighter* copy of the very same image about nine times more
than a genuinely *different* view of the molecule; the FRC is unchanged by brightness, barely moved by filtering, and
drops clearly for a different view. (Filtering changes the squared error little here only because our clean image
holds little power beyond 20 Å; with real images and strong CTF envelopes it matters more.) That is the behaviour we want: a loss that cares about *shape*, not about the microscope's gain or
envelope. Noise lowers the FRC, mostly at high frequencies — which is why the papers fit the low rings first and raise
$r_{\max}$ as the model improves (Section 6).

---
## 6 · Fitting a Gaussian mixture to the particles

Now the central idea of the GMM papers: **skip the voxel map** and fit the Gaussians *directly* to the particle images.

1. Start from a few Gaussians placed at random.
2. Project the GMM at the poses of a batch of particles (Section 4).
3. Score every projection against its particle with the FRC (Section 5).
4. Ask PyTorch for the derivative of the score with respect to every centre, amplitude and width (**autograd**,
   Module 05) and take a gradient step (Adam).
5. When the fit stops improving, **double** the number of Gaussians (split each into two) and raise the resolution limit.

We first rewrite the projection and the FRC in PyTorch so that gradients flow through them.

In [ ]:
# ── 6.1 A differentiable GMM projector and FRC loss in PyTorch ──
GRID_T = torch.tensor((np.arange(N) - (N - 1) / 2) * PX, dtype=torch.float32)
_fy, _fx = np.fft.fftfreq(N), np.fft.rfftfreq(N)
RING_T = torch.tensor(np.minimum((np.sqrt(_fy[:, None] ** 2 + _fx[None, :] ** 2) * N).astype(int), N // 2).ravel())


def project_gmm_torch(centres, amps, sigmas, R):
    """Projections (b, N, N) of a GMM (centres (n, 3) Å, amps (n,), sigmas (n,) Å) at poses R (b, 3, 3)."""
    p = torch.einsum("bij,nj->bni", R, centres)                        # rotated centres, (b, n, 3)
    gx = torch.exp(-(GRID_T - p[..., 0:1]) ** 2 / (2 * sigmas[None, :, None] ** 2))   # (b, n, N)
    gy = torch.exp(-(GRID_T - p[..., 1:2]) ** 2 / (2 * sigmas[None, :, None] ** 2))
    w = amps * math.sqrt(2 * math.pi) * sigmas / PX
    return torch.einsum("bny,bnx->byx", gy * w[None, :, None], gx)


def frc_torch(a, b, rmin: int, rmax: int):
    """Per-image mean FRC over rings rmin..rmax between two batches of images (b, N, N)."""
    Fa = torch.fft.rfft2(a).reshape(a.shape[0], -1)
    Fb = torch.fft.rfft2(b).reshape(b.shape[0], -1)
    ring_sum = lambda v: torch.zeros(a.shape[0], N // 2 + 1, device=v.device).index_add_(1, RING_T, v)
    num = ring_sum((Fa * Fb.conj()).real)
    den = torch.sqrt(ring_sum(Fa.abs() ** 2) * ring_sum(Fb.abs() ** 2) + 1e-12)
    return (num / den)[:, rmin:rmax + 1].mean(1)


# check against the NumPy versions
test_proj = project_gmm_torch(torch.tensor(cen256), torch.tensor(amp256), torch.full((256,), s256),
                              torch.tensor(R_test[None], dtype=torch.float32))[0].numpy()
print(f"PyTorch projector vs gmm_project: max relative difference {np.abs(test_proj - p_analytic).max() / p_analytic.max():.1e}")
noisy_t = torch.tensor(variants["noisy copy (SNR 1)"][None], dtype=torch.float32)
print(f"PyTorch FRC vs NumPy FRC: {frc_torch(torch.tensor(ref[None], dtype=torch.float32), noisy_t, 1, 15).item():.4f}"
      f" vs {frc_score(ref, variants['noisy copy (SNR 1)'], 1, 15):.4f}")

Small differences in the FRC come from the half-plane (`rfft2`) bookkeeping; both measure the same thing.

🧮 **Parameterisation.** We optimise $\log A_j$ and $\log\sigma_j$ rather than $A_j$ and $\sigma_j$, so that amplitudes
and widths stay positive whatever step the optimiser takes. The schedule below — 32 → 64 → 128 → 256 Gaussians, with the
FRC limit moving from 32 Å to 10 Å — is a miniature version of the growing strategy of Chen & Ludtke (2021).

In [ ]:
# ── 6.2 Fit a growing GMM to the 1,000 noisy particles ──
X_t, R_t = torch.tensor(particles), torch.tensor(rot_true)
torch.manual_seed(SEED)
cen_t = torch.tensor(np.random.default_rng(5).normal(0, 15, (32, 3)), dtype=torch.float32)
loga_t, logs_t = torch.zeros(32), torch.full((32,), math.log(8.0))
SCHEDULE = [(32, 32.0, 250), (64, 20.0, 200), (128, 14.0, 200), (256, 10.0, 200)]    # (Gaussians, limit Å, steps)
history, t0 = [], time.time()
for n_g, limit_A, steps in SCHEDULE:
    if len(cen_t) < n_g:                                              # split every Gaussian into two
        jitter = torch.randn_like(cen_t)
        cen_t = torch.cat([cen_t + jitter, cen_t - jitter])
        loga_t, logs_t = torch.cat([loga_t, loga_t]) - math.log(2), torch.cat([logs_t, logs_t]) - 0.2
    params = [p.detach().clone().requires_grad_(True) for p in (cen_t, loga_t, logs_t)]
    cen_t, loga_t, logs_t = params
    opt = torch.optim.Adam([{"params": [cen_t], "lr": 0.5}, {"params": [loga_t, logs_t], "lr": 0.02}])
    rmax = int(N * PX / limit_A)                                      # ring index of the resolution limit
    for it in range(steps):
        idx = torch.randint(0, N_PART, (48,))
        proj = project_gmm_torch(cen_t, loga_t.exp(), logs_t.exp().clamp(1.0, 30.0), R_t[idx])
        loss = -frc_torch(proj, X_t[idx], 1, rmax).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        gm = gmm_to_map(cen_t.numpy(), loga_t.exp().numpy(), logs_t.exp().clamp(1.0, 30.0).numpy(), N, PX)
    f, c, _ = fsc(gm, truth, PX)
    history.append((n_g, limit_A, -loss.item(), resolution_at(f, c, 0.5), gm))
    print(f"{n_g:4d} Gaussians, FRC limit {limit_A:4.0f} Å: training FRC {-loss.item():.3f} → "
          f"map vs truth FSC(0.5) {history[-1][3]:5.1f} Å   [{time.time() - t0:5.1f} s]")
gmm_fit = dict(centres=cen_t.detach().numpy(), amps=loga_t.exp().detach().numpy(),
               sigmas=logs_t.exp().clamp(1.0, 30.0).detach().numpy())

In [ ]:
# ── 6.3 Look at the growing mixture ──
fig, axes = plt.subplots(1, 5, figsize=(12, 2.7))
for ax, (n_g, limit_A, score, res, gm) in zip(axes, history):
    ax.imshow(gm[N // 2]); ax.set_title(f"{n_g} G, {limit_A:.0f} Å limit\n→ {res:.0f} Å", fontsize=8); ax.axis("off")
axes[-1].imshow(truth[N // 2]); axes[-1].set_title("truth", fontsize=8); axes[-1].axis("off")
plt.suptitle("Central slices of the GMM fitted directly to noisy particles", fontsize=10)
plt.tight_layout(); plt.show()

🔍 **What just happened?** From 32 random blobs, gradient descent on the FRC alone moved, brightened, dimmed and resized
the Gaussians until their projections matched the particles. Each doubling let the model express finer detail, and each
raised resolution limit asked it to. We never built a voxel map along the way — the GMM *is* the reconstruction.

⚠️ **Why the poses matter.** We gave the fit the true pose of every particle. In real data the poses come from a
consensus refinement and contain errors; a pose error looks to the fit like a change of shape. Module 26 meets this
problem again when it interprets differences between particles as motion.

---
## 7 · What did the Gaussian mixture buy us?

Section 2 reconstructed a voxel map from the *same* 1,000 particles. Let us put the two side by side with full FSC curves
against the truth.

In [ ]:
# ── 7.1 GMM fit vs direct Fourier reconstruction from the same particles ──
gm_final = history[-1][4]
f, c_gmm, _ = fsc(gm_final, truth, PX)
_, c_dir, _ = fsc(recs[1000], truth, PX)
res_gmm, res_dir = resolution_at(f, c_gmm, 0.5), resolution_at(f, c_dir, 0.5)
n_vox_params, n_gmm_params = N ** 3, 5 * len(gmm_fit["amps"])
print(f"direct reconstruction: {n_vox_params:,} unknowns (voxels) → {res_dir:.1f} Å")
print(f"GMM fit:               {n_gmm_params:,} unknowns (5 per Gaussian) → {res_gmm:.1f} Å")

fig, axes = plt.subplots(1, 4, figsize=(11.5, 2.8), gridspec_kw=dict(width_ratios=[1.6, 1, 1, 1]))
axes[0].plot(f, c_dir, label=f"direct ({res_dir:.0f} Å)"); axes[0].plot(f, c_gmm, label=f"GMM fit ({res_gmm:.0f} Å)")
axes[0].axhline(0.5, color="gray", ls="--", lw=0.8); axes[0].set_xlabel("frequency (1/Å)"); axes[0].set_ylabel("FSC with truth")
axes[0].legend(fontsize=8); axes[0].set_title("same 1,000 particles", fontsize=9)
for ax, im, t in zip(axes[1:], [truth, recs[1000], gm_final], ["truth", "direct", "GMM fit"]):
    ax.imshow(im.sum(axis=0)); ax.set_title(f"{t} (projection)", fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()
assert res_gmm < res_dir, "the GMM fit should beat the unregularised voxel reconstruction here"

🔍 **What just happened?** From identical data, the GMM gives a clearly better map. Two reasons, both worth remembering:

- **Fewer unknowns.** 256 Gaussians are 1,280 numbers; the voxel map has 64,000. Estimating fewer numbers from the same
  noisy data leaves less room to fit noise (Module 04's bias–variance trade-off; Module 10's "Einstein from noise").
- **A built-in prior.** A sum of positive, smooth blobs cannot represent most noise patterns (negative density, salt-and-
  pepper speckle), so the noise that *does* get into the direct map simply has nowhere to go in the GMM.

⚠️ The flip side: a prior can also hide real features (too few or too wide Gaussians smooth away a thin helix) — the
hallucination-and-suppression lesson of Module 24. In practice the papers grow the number of Gaussians until the fit
stops improving and validate with half-maps, as in Module 10.

Most importantly for what follows: every number in the GMM is a **knob** the computer can turn smoothly. Move a group of
centres and the arm swings; dim a group of amplitudes and the partner disappears. Module 26 trains a neural network to
turn those knobs separately for **each particle**.

### 📄 Paper connection — Chen & Ludtke 2021: the deep Gaussian mixture model (e2gmm)

**Paper:** Chen M, Ludtke SJ. *Deep learning-based mixed-dimensional Gaussian mixture model for characterizing
variability in cryo-EM.* Nature Methods 18:930–936 (2021). [doi:10.1038/s41592-021-01220-5](https://doi.org/10.1038/s41592-021-01220-5)
(preprint [arXiv:2101.10356](https://arxiv.org/abs/2101.10356))

- **Question it answers:** how to describe the *variability* among hundreds of thousands of particles — motions and
  changes of composition — in a form that is compact and can be learned from particle images directly.
- **What the paper actually does (the part used here):** the molecule is a GMM; each Gaussian has a 3D centre, an
  amplitude and a width (five parameters). Projections are sums of 2D Gaussians (rotation, z dropped). The fit is scored
  by the FRC averaged over rings, which makes it insensitive to filtering and brightness; images are CTF phase-flipped.
  The number of Gaussians starts at a few hundred at low resolution and is doubled while the projections' FRC stays high,
  in a few rounds (the authors report e.g. ≈ 2,000–3,000 Gaussians for their example maps).
- **What our toy keeps:** GMM, separable projection, ring-normalised FRC loss, autograd fitting with growing $n$.
- **What our toy leaves out:** CTF; the neural network that makes the GMM particle-specific (Module 26); pose refinement;
  GPU-scale data.
- **Code:** EMAN2 (`e2gmm_refine.py`, GUI `e2gmm.py`), <https://github.com/cryoem/eman2>.

### 📄 Paper connection — Chen, Toader & Lederman 2023: scalable, high-resolution deep GMMs

**Paper:** Chen M, Toader B, Lederman R. *Integrating molecular models into CryoEM heterogeneity analysis using scalable
high-resolution deep Gaussian mixture models.* Journal of Molecular Biology 435(9):168014 (2023).
[doi:10.1016/j.jmb.2023.168014](https://doi.org/10.1016/j.jmb.2023.168014) (preprint [arXiv:2211.10518](https://arxiv.org/abs/2211.10518))

- **The parts used here:** the memory-saving projection — each Gaussian is projected on x and y separately, so a
  projection is a product of two $n \times M$ matrices (our Section 4.1) — and the idea of starting the GMM from an
  **atomic model** (one Gaussian per atom), from map pseudo-atoms (K-means on voxels), or both (e.g. for unmodelled
  density such as lipids). The paper's hierarchical "coarse GMM drives fine GMM" design and its rigid-body decoders
  appear in Module 26.

### 📄 Paper connection — Chen, Schmid & Chiu 2024: resolution and resolvability with GMMs

**Paper:** Chen M, Schmid MF, Chiu W. *Improving resolution and resolvability of single-particle cryoEM structures
using Gaussian mixture models.* Nature Methods 21:37–40 (2024). [doi:10.1038/s41592-023-02082-9](https://doi.org/10.1038/s41592-023-02082-9)
(preprint [arXiv:2303.18241](https://arxiv.org/abs/2303.18241))

- **The parts used here:** the GMM and its projection, $P(\mathbf t)=\sum_j A_j e^{-|\mathbf t - R\mathbf c_j|^2/2\sigma_j^2}$
  (up to the constant weight), the FRC loss, and a **model-free** start: K-means on map voxels, with about as many
  Gaussians as the protein has non-hydrogen atoms for near-atomic targets (the authors report e.g. ≈ 7,400 Gaussians for
  a GPCR and ≈ 15,000–17,000 for larger complexes). How the paper then turns per-particle motion into sharper local
  maps is the subject of Module 29.

---
## ✅ Checkpoint

1. A particle image is the 2D Fourier slice of what, through which point, at which angle?
   <details><summary>Answer</summary>Of the molecule's 3D Fourier transform, through its centre (zero frequency), in the plane perpendicular to the beam direction after the particle's rotation: k₃ = Rᵀ(kx, ky, 0).</details>
2. Why does a reconstruction from 20 particles look streaky even without noise?
   <details><summary>Answer</summary>Twenty planes leave most of 3D Fourier space unsampled; the missing coefficients are effectively zero, which produces streaks — the single-particle analogue of the missing wedge (Module 07).</details>
3. What does the projection of a 3D Gaussian of width σ look like, and why is that useful?
   <details><summary>Answer</summary>A 2D Gaussian of the same width centred under the rotated centre, with weight A√(2π)σ. It can be computed exactly and quickly (separably in x and y) and differentiated with respect to every parameter.</details>
4. Give two reasons to prefer the ring-averaged FRC over squared error as a loss.
   <details><summary>Answer</summary>It ignores overall brightness and per-frequency scaling (gain, filtering, CTF envelope) and, by choosing r_max, lets you fit only the frequencies you trust.</details>
5. Our protein has ~5,400 atoms. Roughly how many Gaussians would a 3 Å GMM need, and how many a 20 Å one?
   <details><summary>Answer</summary>Near-atomic: of the order of the number of atoms (thousands). At 20 Å: tens (Section 3: 16 Gaussians gave ≈ 20 Å).</details>
6. Why did the GMM fit beat the direct reconstruction from the same particles?
   <details><summary>Answer</summary>Far fewer unknowns and a positivity/smoothness prior: noise cannot be represented by a few hundred positive blobs, so less of it ends up in the map.</details>

## 🧠 Questions & discussion

1. The GMM fit used the true poses. What would a 5° pose error do to the fitted Gaussians at the edge of a 90 Å protein?
   (Hint: the arc length is radius × angle in radians.)
2. A cryo-EM map often has a lipid nanodisc or detergent micelle around a membrane protein. Should those be part of the
   GMM? What would Chen, Toader & Lederman's "model + pseudo-atoms" start do with them?
3. Why might growing the GMM (32 → 256) work better than starting with 256 random Gaussians? (Think of local minima,
   Module 04's EM.)
4. The FRC ignores per-ring scaling. Can you think of a real change in the molecule that only changes amplitudes per ring
   and would therefore be invisible to the FRC?
5. When is a voxel map *better* than a GMM? (Think of very high resolution, unknown chemistry, or validation.)

## 🚀 Bonus — going further

- **Add a CTF.** Multiply each clean image's Fourier transform by a `ctf_2d` (Module 06; defocus 1–2 µm, pixel 4 Å),
  add noise, then phase-flip (multiply by the sign of the CTF) before fitting. How much worse is the fit?
- **Pose errors.** Perturb every `rot_true` by a random 3° rotation before fitting (Section 6) and before reconstructing
  (Section 2). Which method suffers more?
- **Start from the atoms.** Initialise the GMM with K-means of the *atoms* (cluster the atom coordinates into 256
  groups) instead of random blobs — a miniature of the model-based start of Chen, Toader & Lederman (2023).
- **Real software:** read the documentation of EMAN2's `e2gmm` workflow and identify each of Sections 3–6 in it.

## 📝 Lab note

```text
Module 25 — date:
Input: code-built 3-domain protein (≈ 5,400 atoms), 1,000 projections at random poses, SNR 0.1, 4 Å pixels
Output: Fourier reconstructions (20/100/1,000 particles), K-means GMMs (16–1,024 Gaussians), GMM fitted to particles
Method: projection-slice theorem, trilinear Fourier insertion, separable GMM projection, ring-averaged FRC, Adam
Parameter I changed and what happened:
A failure mode / surprise:
Paper link: Chen & Ludtke 2021; Chen, Toader & Lederman 2023; Chen, Schmid & Chiu 2024
```

## 📚 References

1. Chen M, Ludtke SJ. Deep learning-based mixed-dimensional Gaussian mixture model for characterizing variability in cryo-EM. *Nat Methods* 18:930–936 (2021). doi:10.1038/s41592-021-01220-5
2. Chen M, Toader B, Lederman R. Integrating molecular models into CryoEM heterogeneity analysis using scalable high-resolution deep Gaussian mixture models. *J Mol Biol* 435(9):168014 (2023). doi:10.1016/j.jmb.2023.168014
3. Chen M, Schmid MF, Chiu W. Improving resolution and resolvability of single-particle cryoEM structures using Gaussian mixture models. *Nat Methods* 21:37–40 (2024). doi:10.1038/s41592-023-02082-9
4. Crowther RA, DeRosier DJ, Klug A. The reconstruction of a three-dimensional structure from projections and its application to electron microscopy. *Proc R Soc Lond A* 317:319–340 (1970). doi:10.1098/rspa.1970.0119
5. Penczek PA. Fundamentals of three-dimensional reconstruction from projections. *Methods Enzymol* 482:1–33 (2010). doi:10.1016/S0076-6879(10)82001-4
6. Kawabata T. Multiple subunit fitting into a low-resolution density map of a macromolecular complex using a Gaussian mixture model. *Biophys J* 95:4643–4658 (2008). doi:10.1529/biophysj.108.137125

## ➡️ Next

**[26 · Every particle its own shape: the deep Gaussian mixture model (e2gmm)](26_e2gmm_heterogeneity.ipynb)** — a neural
network reads each particle and moves the Gaussians, so that the arm can swing and the partner can leave.
Course overview: [`README.md`](../README.md) · glossary: [`GLOSSARY.md`](../GLOSSARY.md).

In [ ]:
# ── Total run time ──
print(f"Notebook finished in {time.time() - T_START:.0f} s")